# Euromillions

## Import des librairies

In [1]:
import csv
from datetime import datetime
import io
from IPython.display import display
import numpy as np
import os
import pandas as pd
from pathlib import Path
import re
import requests
import zipfile
import math

## Variables utiles

In [2]:
dossier_destination = Path(r"C:\Users\hbern\Downloads\notebooks\euromillions\csv")
fdj_url = "https://www.sto.api.fdj.fr/anonymous/service-draw-info/v3/documentations"

In [3]:
# Variables utiles
dossier_destination = Path(r"C:\Users\hbern\Downloads\notebooks\euromillions\csv")
fdj_url = "https://www.sto.api.fdj.fr/anonymous/service-draw-info/v3/documentations"

## Import des fichiers

In [4]:
# Liste des fichiers
fichiers = {
    "euromillions": "1a2b3c4d-9876-4562-b3fc-2c963f66afa8",
    "euromillions_2": "1a2b3c4d-9876-4562-b3fc-2c963f66afa9",
    "euromillions_3": "1a2b3c4d-9876-4562-b3fc-2c963f66afb6",
    "euromillions_4": "1a2b3c4d-9876-4562-b3fc-2c963f66afc6", 
    "euromillions_201902": "1a2b3c4d-9876-4562-b3fc-2c963f66afd6",  
    "euromillions_202002": "1a2b3c4d-9876-4562-b3fc-2c963f66afe6"
}

def telecharger_et_dezipper(nom_fichier: str, uuid: str, fdj_url: str, dossier_destination: str | Path):
    dossier_destination = Path(dossier_destination)
    dossier_destination.mkdir(parents=True, exist_ok=True)

    url = f"{fdj_url}/{uuid}"
    print(f"📥 Téléchargement de {url}")

    r = requests.get(url)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        # on prend le premier CSV trouvé dans le ZIP
        csv_names = [n for n in z.namelist() if n.lower().endswith(".csv")]
        if not csv_names:
            raise ValueError(f"Aucun CSV dans l’archive {uuid}")

        interne = csv_names[0]
        z.extract(interne, dossier_destination)

    source = dossier_destination / interne
    destination = dossier_destination / f"{nom_fichier}.csv"

    source.rename(destination)
    print(f"✅ {destination.name} créé")

for nom_fichier, uuid in fichiers.items():
    fp = dossier_destination / f"{nom_fichier}.csv"

    # cas spécial : toujours réimporter le dernier fichier
    if nom_fichier == "euromillions_202002":
        print(f"♻️ Réimport forcé de {fp.name}")
        # telecharger_et_dezipper(nom_fichier, uuid, fdj_url, dossier_destination)
        continue

    # comportement normal pour les autres
    if fp.exists():
        print(f"⏭️ {fp.name} existe déjà")
        continue

    print(f"📂 {fp.name} absent → import")
    
    # telecharger_et_dezipper(nom_fichier, uuid, fdj_url, dossier_destination)

⏭️ euromillions.csv existe déjà
⏭️ euromillions_2.csv existe déjà
⏭️ euromillions_3.csv existe déjà
⏭️ euromillions_4.csv existe déjà
⏭️ euromillions_201902.csv existe déjà
♻️ Réimport forcé de euromillions_202002.csv


## Fonction de lecture des fichiers 

In [5]:
def lire_csv(file: str, dossier_destination: str) -> pd.DataFrame:
    """Lit un fichier CSV et renvoie un DataFrame pandas."""

    path = r"C:\Users\hbern\Downloads\notebooks\euromillions\csv" # Chemin vers répertoire de stockage des fichiers csv
    file_path = path+"\\"+file

    # print("file_path : "+file_path)
    
    encoding_type = "utf-8-sig"

    if file == "euromillions_4.csv":
        encoding_type = "cp1252"
    
    if file == "euromillions_201902.csv":
        encoding_type = "cp1252"
    
    df = pd.read_csv(
        file_path,
        index_col=False,
        engine="python",
        sep=None,
        encoding=encoding_type,
        quotechar='"',
        escapechar='\\',
        skip_blank_lines=True,
        on_bad_lines="skip"
    )    

    # Remet les dates de tirage par ordre croissant, sauf le fichier euromillions_4 où elles le sont déjà
    if file != "euromillions_4.csv":
        df = df.iloc[::-1].reset_index(drop=True)
    
    return df


## Colonnes

In [6]:
# Liste des colonnes à supprimer dans les fichiers

# fichier 1
colonnes_a_supprimer = ['date_de_forclusion', 'nombre_de_gagnant_au_rang2_en_france', 'nombre_de_gagnant_au_rang2_en_europe', 'rapport_du_rang2',
       'nombre_de_gagnant_au_rang3_en_france', 'nombre_de_gagnant_au_rang3_en_europe', 'rapport_du_rang3', 'nombre_de_gagnant_au_rang4_en_france', 'nombre_de_gagnant_au_rang4_en_europe', 'rapport_du_rang4',
       'nombre_de_gagnant_au_rang5_en_france', 'nombre_de_gagnant_au_rang5_en_europe', 'rapport_du_rang5', 'nombre_de_gagnant_au_rang6_en_france', 'nombre_de_gagnant_au_rang6_en_europe', 'rapport_du_rang6',
       'nombre_de_gagnant_au_rang7_en_france', 'nombre_de_gagnant_au_rang7_en_europe', 'rapport_du_rang7', 'nombre_de_gagnant_au_rang8_en_france', 'nombre_de_gagnant_au_rang8_en_europe', 'rapport_du_rang8',
       'nombre_de_gagnant_au_rang9_en_france', 'nombre_de_gagnant_au_rang9_en_europe', 'rapport_du_rang9', 'nombre_de_gagnant_au_rang10_en_france', 'nombre_de_gagnant_au_rang10_en_europe', 'rapport_du_rang10',
       'nombre_de_gagnant_au_rang11_en_france', 'nombre_de_gagnant_au_rang11_en_europe', 'rapport_du_rang11', 'nombre_de_gagnant_au_rang12_en_france', 'nombre_de_gagnant_au_rang12_en_europe', 'rapport_du_rang12',
       'numero_jokerplus', 'devise', 'Unnamed: 51']

# fichier 2
colonnes_a_supprimer += ['nombre_de_gagnant_au_rang13_en_france', 'nombre_de_gagnant_au_rang13_en_europe', 'rapport_du_rang13', 'Unnamed: 54']

# fichier 3
colonnes_a_supprimer += ['numero_My_Million']

# fichier 4
colonnes_a_supprimer += ['numéro_de_tirage_dans_le_cycle', 'nombre_de_gagnant_au_rang2_Euro_Millions_en_france', 'nombre_de_gagnant_au_rang2_Euro_Millions_en_europe', 'rapport_du_rang2_Euro_Millions', 'nombre_de_gagnant_au_rang3_Euro_Millions_en_france', 'nombre_de_gagnant_au_rang3_Euro_Millions_en_europe', 'rapport_du_rang3_Euro_Millions', 'nombre_de_gagnant_au_rang4_Euro_Millions_en_france', 'nombre_de_gagnant_au_rang4_Euro_Millions_en_europe', 'rapport_du_rang4_Euro_Millions', 'nombre_de_gagnant_au_rang5_Euro_Millions_en_france', 'nombre_de_gagnant_au_rang5_Euro_Millions_en_europe', 'rapport_du_rang5_Euro_Millions', 'nombre_de_gagnant_au_rang6_Euro_Millions_en_france', 'nombre_de_gagnant_au_rang6_Euro_Millions_en_europe', 'rapport_du_rang6_Euro_Millions', 'nombre_de_gagnant_au_rang7_Euro_Millions_en_france', 'nombre_de_gagnant_au_rang7_Euro_Millions_en_europe', 'rapport_du_rang7_Euro_Millions', 'nombre_de_gagnant_au_rang8_Euro_Millions_en_france', 'nombre_de_gagnant_au_rang8_Euro_Millions_en_europe', 'rapport_du_rang8_Euro_Millions', 'nombre_de_gagnant_au_rang9_Euro_Millions_en_france', 'nombre_de_gagnant_au_rang9_Euro_Millions_en_europe', 'rapport_du_rang9_Euro_Millions', 'nombre_de_gagnant_au_rang10_Euro_Millions_en_france', 'nombre_de_gagnant_au_rang10_Euro_Millions_en_europe', 'rapport_du_rang10_Euro_Millions', 'nombre_de_gagnant_au_rang11_Euro_Millions_en_france', 'nombre_de_gagnant_au_rang11_Euro_Millions_en_europe', 'rapport_du_rang11_Euro_Millions', 'nombre_de_gagnant_au_rang12_Euro_Millions_en_france', 'nombre_de_gagnant_au_rang12_Euro_Millions_en_europe', 'rapport_du_rang12_Euro_Millions', 'nombre_de_gagnant_au_rang13_Euro_Millions_en_france', 'nombre_de_gagnant_au_rang13_Euro_Millions_en_europe', 'rapport_du_rang13_Euro_Millions', 'nombre_de_gagnant_au_rang1_Etoile+', 'rapport_du_rang1_Etoile+', 'nombre_de_gagnant_au_rang2_Etoile+', 'rapport_du_rang2_Etoile+', 'nombre_de_gagnant_au_rang3_Etoile+', 'rapport_du_rang3_Etoile+', 'nombre_de_gagnant_au_rang4_Etoile+', 'rapport_du_rang4_Etoile+',
       'nombre_de_gagnant_au_rang5_Etoile+', 'rapport_du_rang5_Etoile+', 'nombre_de_gagnant_au_rang6_Etoile+', 'rapport_du_rang6_Etoile+', 'nombre_de_gagnant_au_rang7_Etoile+', 'rapport_du_rang7_Etoile+', 'nombre_de_gagnant_au_rang8_Etoile+', 'rapport_du_rang8_Etoile+', 'nombre_de_gagnant_au_rang9_Etoile+', 'rapport_du_rang9_Etoile+', 'nombre_de_gagnant_au_rang10_Etoile+', 'rapport_du_rang10_Etoile+', 'numero_Tirage_Exceptionnel_Euro_Millions']

# fichier 201902
colonnes_a_supprimer += ['Unnamed: 75']

# fichier 202002
colonnes_a_supprimer += ['numero_Tirage_Exceptionnel_Euro_Million']

# print(colonnes_a_supprimer)
# print("Nombre de colonnes à supprimer : " + str(len(colonnes_a_supprimer)))
# Dédoublonne pour être sûr 
# colonnes_a_supprimer = list(set(colonnes_a_supprimer))
# print("Nombre de colonnes à supprimer (dédoublonnée) : " + str(len(colonnes_a_supprimer)))

## 1. Fichier euromillions.csv

- 13/02/2004 au 06/05/2011
- Tirages 1 à 378 (indices 0 à 377)

In [7]:
df1 = lire_csv("euromillions.csv", dossier_destination)

# Supprime les colonnes inutiles 
df1 = df1.drop(columns=colonnes_a_supprimer, errors="ignore")

# Date tirage : colonne 'date_de_tirage', format : AAAAMMJJ 
# Formate à JJ/MM/AAAA 
df1["date_de_tirage"] = pd.to_datetime(df1["date_de_tirage"], format="%Y%m%d")
df1["date_de_tirage"] = df1["date_de_tirage"].dt.strftime("%d/%m/%Y")

# Affiche les 5 1ères + 5 dernières lignes
# df1.head(df1.shape[0]) # 378,15

## 2. Fichier euromillions_2.csv

- 10/05/2011 au 31/01/2014
- Tirages 379 à 664

In [8]:
df2 = lire_csv("euromillions_2.csv", dossier_destination)

# Supprime les colonnes inutiles 
df2 = df2.drop(columns=colonnes_a_supprimer, errors="ignore")

# Affiche les 5 1ères + 5 dernières lignes
# df2.head(df2.shape[0]) # (286, 15)

## 3. Fichier euromillions_3.csv

- 04/02/2014 au 23/09/2016
- Tirages 665 à 940
- Introduction numéro My Million

In [9]:
df3 = lire_csv("euromillions_3.csv", dossier_destination)

# Supprime les colonnes inutiles 
df3 = df3.drop(columns=colonnes_a_supprimer, errors="ignore")

# Corrige en 23/09/2016 la date de la dernière ligne :
# 275 	2016077 	VENDREDI 	23/09/16 	
df3.loc[275, "date_de_tirage"] = "23/09/2016"

# Affiche les 5 1ères + 5 dernières lignes
# df3.head(df3.shape[0]) # (276, 15)

## 4. Fichier euromillions_4.csv

- 27/09/2016 au 26/02/2019
- Tirages 941 à 1290 
- Introduction Etoile+

In [10]:
df4 = lire_csv("euromillions_4.csv", dossier_destination)

# Supprime les colonnes inutiles 
df4 = df4.drop(columns=colonnes_a_supprimer, errors="ignore")

# Affiche les 5 1ères + 5 dernières lignes
# df4.head(df4.shape[0]) # (253, 15)

## 5. Fichier euromillions_201902.csv

- 01/03/2019 au 31/01/2020
- Tirages 1194 à 1290  

In [11]:
df5 = lire_csv("euromillions_201902.csv", dossier_destination)

# Supprime les colonnes inutiles 
df5 = df5.drop(columns=colonnes_a_supprimer, errors="ignore")

# Affiche les 5 1ères + 5 dernières lignes
# df5.head(df5.shape[0]) # (97, 15)

## 6. Fichier euromillions_202002.csv

- Depuis le 04/02/2020
- Depuis tirage 1291   

In [12]:
def telecharger_et_dezipper(url: str, dossier_destination: str | Path):
    """
        Télécharge un fichier ZIP depuis une URL et le décompresse dans le dossier indiqué.
    """
    dossier_destination = Path(dossier_destination)
    dossier_destination.mkdir(parents=True, exist_ok=True)  # crée le dossier si besoin

    print(f"📥 Téléchargement de {url} ...")
    r = requests.get(url)
    r.raise_for_status()  # lève une erreur si le téléchargement échoue

    print("🗜️ Décompression...")
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extractall(dossier_destination)

    print(f"✅ Fichiers extraits dans : {dossier_destination.resolve()}")

telecharger_et_dezipper("https://www.sto.api.fdj.fr/anonymous/service-draw-info/v3/documentations/1a2b3c4d-9876-4562-b3fc-2c963f66afe6", dossier_destination)

df6 = lire_csv("euromillions_202002.csv", dossier_destination)

# Supprime les colonnes inutiles 
df6 = df6.drop(columns=colonnes_a_supprimer, errors="ignore")

df6.head(df6.shape[0]) # (XXX, 15)

print(f"Date du dernier tirage : {df6["date_de_tirage"].iloc[-1]}")

📥 Téléchargement de https://www.sto.api.fdj.fr/anonymous/service-draw-info/v3/documentations/1a2b3c4d-9876-4562-b3fc-2c963f66afe6 ...
🗜️ Décompression...
✅ Fichiers extraits dans : C:\Users\hbern\Downloads\notebooks\euromillions\csv
Date du dernier tirage : 08/05/2026


In [13]:
colonnes_nouvelles = ["annee_numero_tirage", "jour_tirage", "date_tirage", 
               "boule_1", "boule_2", "boule_3", "boule_4", "boule_5",
               "etoile_1", "etoile_2",
               "boules_croissant", "etoiles_croissant",
               "nb_gagnants_r1_fr", "nb_gagnants_r1_eu", "rapport_r1"
              ]

# Liste des DataFrames
dataframes = [df1, df2, df3, df4, df5, df6]

# Boucle sur la liste
for df in dataframes:
    df.columns = colonnes_nouvelles

## Dernier tirage

In [14]:
df6.tail(1)

,annee_numero_tirage,jour_tirage,date_tirage,boule_1,boule_2,boule_3,boule_4,boule_5,etoile_1,etoile_2,boules_croissant,etoiles_croissant,nb_gagnants_r1_fr,nb_gagnants_r1_eu,rapport_r1
653,26037,VENDREDI,08/05/2026,19,17,37,34,2,11,8,-2-17-19-34-37-,-8-11-,0,0,0


## Les 10 derniers tirages

In [15]:
df6.tail(10)

,annee_numero_tirage,jour_tirage,date_tirage,boule_1,boule_2,boule_3,boule_4,boule_5,etoile_1,etoile_2,boules_croissant,etoiles_croissant,nb_gagnants_r1_fr,nb_gagnants_r1_eu,rapport_r1
644,26028,MARDI,07/04/2026,14,49,11,19,36,7,6,-11-14-19-36-49-,-6-7-,0,0,0
645,26029,VENDREDI,10/04/2026,10,13,41,38,14,6,9,-10-13-14-38-41-,-6-9-,0,0,0
646,26030,MARDI,14/04/2026,1,2,4,44,28,12,5,-1-2-4-28-44-,-5-12-,0,0,0
647,26031,VENDREDI,17/04/2026,22,23,47,28,41,6,8,-22-23-28-41-47-,-6-8-,0,0,0
648,26032,MARDI,21/04/2026,29,47,16,13,40,4,3,-13-16-29-40-47-,-3-4-,1,3,"48175066,00"
649,26033,VENDREDI,24/04/2026,40,30,45,26,25,5,1,-25-26-30-40-45-,-1-5-,0,0,0
650,26034,MARDI,28/04/2026,46,47,29,26,41,9,8,-26-29-41-46-47-,-8-9-,0,0,0
651,26035,VENDREDI,01/05/2026,42,47,46,3,9,11,1,-3-9-42-46-47-,-1-11-,0,0,0
652,26036,MARDI,05/05/2026,4,8,3,31,20,6,8,-3-4-8-20-31-,-6-8-,0,0,0
653,26037,VENDREDI,08/05/2026,19,17,37,34,2,11,8,-2-17-19-34-37-,-8-11-,0,0,0


In [16]:
# Concaténation
df = pd.concat([df1, df2, df3, df4, df5, df6], ignore_index=True)

# df.head(df.shape[0]) 

In [17]:
# Etat du dataframe
df.dtypes
df.info()

df.isna()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1944 entries, 0 to 1943
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   annee_numero_tirage  1944 non-null   int64 
 1   jour_tirage          1944 non-null   object
 2   date_tirage          1944 non-null   object
 3   boule_1              1944 non-null   int64 
 4   boule_2              1944 non-null   int64 
 5   boule_3              1944 non-null   int64 
 6   boule_4              1944 non-null   int64 
 7   boule_5              1944 non-null   int64 
 8   etoile_1             1944 non-null   int64 
 9   etoile_2             1944 non-null   int64 
 10  boules_croissant     1944 non-null   object
 11  etoiles_croissant    1944 non-null   object
 12  nb_gagnants_r1_fr    1944 non-null   int64 
 13  nb_gagnants_r1_eu    1944 non-null   int64 
 14  rapport_r1           1944 non-null   object
dtypes: int64(10), object(5)
memory usage: 227.9+ KB


annee_numero_tirage    0
jour_tirage            0
date_tirage            0
boule_1                0
boule_2                0
boule_3                0
boule_4                0
boule_5                0
etoile_1               0
etoile_2               0
boules_croissant       0
etoiles_croissant      0
nb_gagnants_r1_fr      0
nb_gagnants_r1_eu      0
rapport_r1             0
dtype: int64

In [18]:
# jour tirage = 'V' ou 'M' 
df["jour_tirage"] = df["jour_tirage"].astype(str).str[0]

# supprime les '-' en début et fin 
df["boules_croissant"] = df["boules_croissant"].astype(str).str.strip("-")
df["etoiles_croissant"] = df["etoiles_croissant"].astype(str).str.strip("-")

df.head(df.shape[0]) # 1893 rows × 15 columns

# Démarre l'index à 1 au lieu de 0 = correspond ainsi au numéro de tirage depuis le début
df = df.reset_index(drop=True)
df.index += 1

# Conversion de la colonne 'date' en datetime
df["date_tirage"] = pd.to_datetime(df["date_tirage"], dayfirst=True, errors="coerce")

# Format JJ/MM/AAAA
df["date_tirage"] = df["date_tirage"].dt.strftime("%d/%m/%Y")  

In [19]:
# Export du dataframe global

# on supprime le fichier 'csv/global.csv' sil existe 
global_file_path = os.path.join(dossier_destination, "global.csv")

if os.path.exists(global_file_path):
    os.remove(global_file_path)

df.to_csv(global_file_path, index=False, encoding="utf-8-sig")

## Statistiques globales

In [20]:
# -----------------------------------------------------------------------
# 1) nettoyage
# -----------------------------------------------------------------------

# Normalisation des types
# - Conversion des colonnes numériques mal typées
for col in ["etoile_2", "rapport_r1"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Conversion de la date
# if "date_tirage" in df.columns:
#    df["date_tirage"] = pd.to_datetime(df["date_tirage"], errors="coerce", format="%Y%m%d").fillna(pd.to_datetime(df["date_tirage"], errors="coerce"))

In [21]:
# -*- coding: utf-8 -*-
# === EuroMillions : tableaux colorés (Styler) ===
# Prérequis : pandas, numpy
import numpy as np
from IPython.display import display, HTML

# -----------------------------------------------------------------------
# 1) nettoyage
# -----------------------------------------------------------------------

for col in ["etoile_2", "rapport_r1"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "date_tirage" in df.columns:
    df["date_tirage"] = pd.to_datetime(df["date_tirage"], dayfirst=True, errors="coerce", format="%Y%m%d").fillna(
        pd.to_datetime(df["date_tirage"], dayfirst=True, errors="coerce")
    )
    df["annee"] = df["date_tirage"].dt.year
    df["mois"]  = df["date_tirage"].dt.to_period("M").astype(str)

BOULES  = [c for c in df.columns if c.startswith("boule_")]
ETOILES = [c for c in df.columns if c.startswith("etoile_")]

print("Nb de tirages :", len(df))
print("Colonnes boules :", BOULES)
print("Colonnes étoiles :", ETOILES)

# -----------------------------------------------------------------------
# 2) Descriptifs par position (tableaux simples)
# -----------------------------------------------------------------------
desc_boules  = df[BOULES].agg(["mean", "median", "std", "min", "max"]).T.round(2)
desc_etoiles = df[ETOILES].agg(["mean", "median", "std", "min", "max"]).T.round(2)

print("\n=== Descriptifs boules (par position) ===")
display(desc_boules)
print("\n=== Descriptifs étoiles (par position) ===")
display(desc_etoiles)

# -----------------------------------------------------------------------
# 3) Fréquences globales d’apparition (toutes positions confondues)
# -----------------------------------------------------------------------
vals_boules = pd.Series(pd.to_numeric(df[BOULES].values.ravel(), errors="coerce"), name="boule").dropna().astype(int)
freq_boules = vals_boules.value_counts().sort_index()
freq_boules = freq_boules.reindex(pd.Index(range(1, 51), name="boule"), fill_value=0)

vals_etoiles = pd.Series(pd.to_numeric(df[ETOILES].values.ravel(), errors="coerce"), name="etoile").dropna().astype(int)
freq_etoiles = vals_etoiles.value_counts().sort_index()
freq_etoiles = freq_etoiles.reindex(pd.Index(range(1, 13), name="etoile"), fill_value=0)

tab_boules = pd.DataFrame({
    "frequence": freq_boules,
    "freq_par_tirage": freq_boules / len(df),
    "prob_empirique": freq_boules / len(df)
}).sort_values("frequence", ascending=False)

tab_etoiles = pd.DataFrame({
    "frequence": freq_etoiles,
    "freq_par_tirage": freq_etoiles / len(df),
    "prob_empirique": freq_etoiles / len(df)
}).sort_values("frequence", ascending=False)

# -----------------------------------------------------------------------
# 4) Styles colorés (barres et dégradés)
# -----------------------------------------------------------------------
def style_freq_table(tab, titre):
    # Colonnes formatées
    fmt = {
        "frequence": "{:,.0f}",
        "freq_par_tirage": "{:.6f}",
        "prob_empirique": "{:.6f}",
    }
    # Styler : barre horizontale sur 'frequence', dégradé sur 'prob_empirique'
    sty = (
        tab.style
        .format(fmt)
        .bar(subset=["frequence"], align="left")         # barre de progression
        .background_gradient(subset=["prob_empirique"])  # dégradé
        .set_table_attributes('style="width:100%; border-collapse:collapse"')
        .set_caption(titre)
        .set_properties(**{"text-align": "right"})
    )
    # Index à gauche
    sty = sty.set_properties(subset=pd.IndexSlice[:, :], **{"text-align": "right"})
    return sty

print("\n=== Tableau coloré : Boules (classées par fréquence) ===")
display(style_freq_table(tab_boules, "Boules les plus fréquentes (toutes positions confondues)"))

print("\n=== Tableau coloré : Étoiles (classées par fréquence) ===")
display(style_freq_table(tab_etoiles, "Étoiles les plus fréquentes (toutes positions confondues)"))

# -----------------------------------------------------------------------
# 7) Export HTML des tableaux stylés (pratique pour partager)
# -----------------------------------------------------------------------
# Remarque : .to_html() sur un Styler nécessite pandas >= 1.3
boules_html  = style_freq_table(tab_boules, "Boules les plus fréquentes").to_html()
etoiles_html = style_freq_table(tab_etoiles, "Étoiles les plus fréquentes").to_html()
Path("table_boules.html").write_text(boules_html, encoding="utf-8")
Path("table_etoiles.html").write_text(etoiles_html, encoding="utf-8")
print("\n✅ Export HTML effectué : table_boules.html, table_etoiles.html")

# -----------------------------------------------------------------------
# 8) Comparaison simple avec la théorie (rappel)
# -----------------------------------------------------------------------
theorie_boules  = 5/50
theorie_etoiles = 2/12
print("\n=== Comparaison théorie vs empirique (moyenne des probas empiriques) ===")
print("Théorie (boule)  ≈", round(theorie_boules, 6),
      "| Empirique (moyenne) :", round(tab_boules['prob_empirique'].mean(), 6))
print("Théorie (étoile) ≈", round(theorie_etoiles, 6),
      "| Empirique (moyenne) :", round(tab_etoiles['prob_empirique'].mean(), 6))

print("\n🎯 Astuce : clique sur les en-têtes pour trier (dans Jupyter), ou ouvre les HTML exportés pour partager.")


Nb de tirages : 1944
Colonnes boules : ['boule_1', 'boule_2', 'boule_3', 'boule_4', 'boule_5']
Colonnes étoiles : ['etoile_1', 'etoile_2']

=== Descriptifs boules (par position) ===


,mean,median,std,min,max
boule_1,25.55,26.0,14.39,1.0,50.0
boule_2,25.17,25.0,14.35,1.0,50.0
boule_3,26.14,26.0,14.33,1.0,50.0
boule_4,25.37,25.0,14.47,1.0,50.0
boule_5,25.30,25.0,14.34,1.0,50.0



=== Descriptifs étoiles (par position) ===


,mean,median,std,min,max
etoile_1,6.14,6.0,3.28,1.0,12.0
etoile_2,6.01,6.0,3.27,1.0,12.0



=== Tableau coloré : Boules (classées par fréquence) ===


,frequence,freq_par_tirage,prob_empirique
boule,,,
44,222,0.114198,0.114198
42,220,0.113169,0.113169
23,218,0.112140,0.112140
19,217,0.111626,0.111626
29,216,0.111111,0.111111
17,212,0.109053,0.109053
21,212,0.109053,0.109053
10,211,0.108539,0.108539
50,209,0.107510,0.107510



=== Tableau coloré : Étoiles (classées par fréquence) ===


,frequence,freq_par_tirage,prob_empirique
etoile,,,
3,384,0.197531,0.197531
2,383,0.197016,0.197016
8,371,0.190844,0.190844
9,358,0.184156,0.184156
6,350,0.180041,0.180041
5,345,0.177469,0.177469
7,344,0.176955,0.176955
1,334,0.171811,0.171811
4,307,0.157922,0.157922



✅ Export HTML effectué : table_boules.html, table_etoiles.html

=== Comparaison théorie vs empirique (moyenne des probas empiriques) ===
Théorie (boule)  ≈ 0.1 | Empirique (moyenne) : 0.1
Théorie (étoile) ≈ 0.166667 | Empirique (moyenne) : 0.166667

🎯 Astuce : clique sur les en-têtes pour trier (dans Jupyter), ou ouvre les HTML exportés pour partager.


In [22]:
# Y-a-t-il une combinaison sortie plus d'une fois (improbable mais il y en a bien !)
# keep=False permet d'afficher toutes les occurences (sans : n'affiche que la 1ère trouvée)
df[df["boules_croissant"].duplicated(keep=False)]

,annee_numero_tirage,jour_tirage,date_tirage,boule_1,boule_2,boule_3,boule_4,boule_5,etoile_1,etoile_2,boules_croissant,etoiles_croissant,nb_gagnants_r1_fr,nb_gagnants_r1_eu,rapport_r1,annee,mois
690,2014035,V,2014-05-02,30,42,4,31,38,11,2,4-30-31-38-42,2-11,0,1,89166455.0,2014,2014-05
1142,2018070,V,2018-08-31,4,42,30,38,31,4,6,4-30-31-38-42,4-6,0,0,0.0,2018,2018-08


In [23]:
# --- 2) Parsing robuste des dates (mix 20040213 et 30/12/2025) ---
# s = df["date_tirage"].astype(str).str.strip()
# mask8 = s.str.fullmatch(r"\d{8}")

# d1 = pd.to_datetime(s.where(mask8), format="%Y%m%d", errors="coerce")
# d2 = pd.to_datetime(s.where(~mask8), dayfirst=True, errors="coerce")
# df["date_tirage"] = d1.fillna(d2)

df = df.sort_values("date_tirage").reset_index(drop=True)
df["draw_id"] = np.arange(len(df))

# --- 3) Colonnes boules / étoiles ---
ball_cols = [f"boule_{i}" for i in range(1, 6)]
star_cols = [f"etoile_{i}" for i in range(1, 3)]

# --- 4) Fonction générique de stats ---
def build_stats(df, cols, universe_size, picks_per_draw):
    m = (
        df[["draw_id", "date_tirage"] + cols]
        .melt(id_vars=["draw_id", "date_tirage"], value_name="numero")
        .dropna()
    )
    m["numero"] = m["numero"].astype(int)

    nb_draws = df["draw_id"].nunique()
    last_draw_id = df["draw_id"].max()
    last_date = df["date_tirage"].max()

    m_sorted = m.sort_values(["numero", "draw_id"])

    # nb sorties
    out = m_sorted.groupby("numero").size().rename("nb_sorties").reset_index()

    # dernière occurrence
    last = (
        m_sorted.groupby("numero").tail(1)[["numero", "draw_id", "date_tirage"]]
        .rename(columns={"draw_id": "dernier_draw_id", "date_tirage": "dernier_date"})
    )

    # occurrence précédente (avant-dernière)
    prev = (
        m_sorted.groupby("numero").nth(-2).reset_index()[["numero", "draw_id", "date_tirage"]]
        .rename(columns={"draw_id": "precedent_draw_id", "date_tirage": "precedent_date"})
    )

    out = out.merge(last, on="numero", how="left").merge(prev, on="numero", how="left")

    # écarts
    out["ecart_tirages"] = out["dernier_draw_id"] - out["precedent_draw_id"]
    out.loc[out["precedent_draw_id"].isna(), "ecart_tirages"] = np.nan

    out["ecart_depuis_dernier"] = last_draw_id - out["dernier_draw_id"]
    out["jours_depuis_dernier"] = (last_date - out["dernier_date"]).dt.days

    # fréquence observée
    out["freq_par_tirage"] = out["nb_sorties"] / nb_draws

    # proba théorique qu’un numéro apparaisse au prochain tirage
    out["proba_theorique"] = picks_per_draw / universe_size

    # moyenne historique de l'écart (en nombre de tirages) entre sorties
    moy_gap = (
        m_sorted.groupby("numero")["draw_id"]
        .diff()
        .groupby(m_sorted["numero"])
        .mean()
        .rename("moy_ecart_tirages")
        .reset_index()
    )
    out = out.merge(moy_gap, on="numero", how="left")

    # proba heuristique: base * ajustement "retard / moyenne"
    # (clamp léger pour éviter les valeurs absurdes)
    ratio = out["ecart_depuis_dernier"] / out["moy_ecart_tirages"]
    out["proba_heuristique"] = out["proba_theorique"] * (1 + (ratio - 1).clip(lower=-1, upper=3) * 0.25)
    out["proba_heuristique"] = out["proba_heuristique"].clip(0, 1)

    # garantir tous les numéros de l'univers
    all_nums = pd.DataFrame({"numero": np.arange(1, universe_size + 1)})
    out = all_nums.merge(out, on="numero", how="left")
    out["nb_sorties"] = out["nb_sorties"].fillna(0).astype(int)

    return out.sort_values("numero").reset_index(drop=True)

# --- 5) DataFrames finaux ---
df_boules = build_stats(df, ball_cols, universe_size=50, picks_per_draw=5)

stars_max = int(df[star_cols].max().max())  # s'adapte si 11/12 selon l'historique
df_etoiles = build_stats(df, star_cols, universe_size=stars_max, picks_per_draw=2)

# --- 6) Exemples d'usage ---
# Top 10 "heuristique" (boules)
# top10_boules = df_boules.sort_values("proba_heuristique", ascending=False).head(10)

# Top 10 "heuristique" (étoiles)
#top10_etoiles = df_etoiles.sort_values("proba_heuristique", ascending=False).head(10)

#print(top10_boules[["numero","nb_sorties","ecart_depuis_dernier","moy_ecart_tirages","proba_theorique","proba_heuristique"]])
#print(top10_etoiles[["numero","nb_sorties","ecart_depuis_dernier","moy_ecart_tirages","proba_theorique","proba_heuristique"]])

display(df_boules)



,numero,nb_sorties,dernier_draw_id,dernier_date,precedent_draw_id,precedent_date,ecart_tirages,ecart_depuis_dernier,jours_depuis_dernier,freq_par_tirage,proba_theorique,moy_ecart_tirages,proba_heuristique
0,1,185,1936,2026-04-14,1920,2026-02-17,16.0,7,24,0.095165,0.1,10.478261,0.091701
1,2,180,1943,2026-05-08,1936,2026-04-14,7.0,0,0,0.092593,0.1,10.759777,0.075000
2,3,189,1942,2026-05-05,1941,2026-05-01,1.0,1,3,0.097222,0.1,10.297872,0.077428
3,4,198,1942,2026-05-05,1936,2026-04-14,6.0,1,3,0.101852,0.1,9.842640,0.077540
4,5,198,1932,2026-03-31,1929,2026-03-20,3.0,11,38,0.101852,0.1,9.751269,0.103201
5,6,188,1924,2026-03-03,1920,2026-02-17,4.0,19,66,0.096708,0.1,10.235294,0.121408
6,7,198,1924,2026-03-03,1909,2026-01-09,15.0,19,66,0.101852,0.1,9.761421,0.123661
7,8,186,1942,2026-05-05,1933,2026-04-03,9.0,1,3,0.095679,0.1,10.405405,0.077403
8,9,184,1941,2026-05-01,1919,2026-02-13,22.0,2,7,0.094650,0.1,10.519126,0.079753
9,10,211,1935,2026-04-10,1932,2026-03-31,3.0,8,28,0.108539,0.1,9.185714,0.096773


In [24]:
df = df.sort_values("date_tirage").reset_index(drop=True)

ball_cols = [f"boule_{i}" for i in range(1, 6)]

# Dirichlet
def bayesian_scores(df, cols, universe_size, alpha=1.0):
    draws = df[cols].values.flatten()
    draws = draws[~pd.isna(draws)].astype(int)

    counts = pd.Series(draws).value_counts().reindex(
        range(1, universe_size + 1), fill_value=0
    )

    n_draws = df.shape[0]
    picks = len(cols)

    # a posteriori Dirichlet
    posterior = (counts + alpha) / (n_draws * picks + alpha * universe_size)

    # proba théorique
    proba_uniforme = 1 / universe_size

    result = pd.DataFrame({
        "numero": counts.index,
        "nb_sorties": counts.values,
        "proba_bayesienne": posterior.values,
        "ratio_vs_uniforme": posterior.values / proba_uniforme
    })

    return result.sort_values("proba_bayesienne", ascending=False)

df_bayes_boules = bayesian_scores(
    df,
    ball_cols,
    universe_size=50,
    alpha=1
)

display(df_bayes_boules.head(10))

,numero,nb_sorties,proba_bayesienne,ratio_vs_uniforme
43,44,222,0.022825,1.141249
41,42,220,0.022620,1.131013
22,23,218,0.022416,1.120778
18,19,217,0.022313,1.115660
28,29,216,0.022211,1.110542
16,17,212,0.021801,1.090072
20,21,212,0.021801,1.090072
9,10,211,0.021699,1.084954
49,50,209,0.021494,1.074719
36,37,206,0.021187,1.059365


In [25]:

def bayesian_scores(df, cols, universe_size, alpha=1.0):
    values = df[cols].values.flatten()
    values = values[~pd.isna(values)].astype(int)

    counts = pd.Series(values).value_counts().reindex(
        range(1, universe_size + 1), fill_value=0
    )

    n_draws = df.shape[0]
    picks = len(cols)

    posterior = (counts + alpha) / (n_draws * picks + alpha * universe_size)

    proba_uniforme = 1 / universe_size

    return pd.DataFrame({
        "numero": counts.index,
        "nb_sorties": counts.values,
        "proba_bayesienne": posterior.values,
        "ratio_vs_uniforme": posterior.values / proba_uniforme
    }).sort_values("proba_bayesienne", ascending=False)

ball_cols = [f"boule_{i}" for i in range(1, 6)]

df_boules = bayesian_scores(
    df,
    ball_cols,
    universe_size=50,
    alpha=1
)

star_cols = [f"etoile_{i}" for i in range(1, 3)]

df_etoiles = bayesian_scores(
    df,
    star_cols,
    universe_size=12,
    alpha=1
)

# Top 12 boules bayésiennes
top_boules = df_boules.head(12)

# Choix équilibré (1 sur 2)
boules_finales = top_boules.iloc[[0, 2, 4, 6, 8]]["numero"].sort_values().tolist()



In [26]:
# générer 3 ou 5 grilles différentes (diversification)



# df_boules et df_etoiles sont les DataFrames bayésiens
# issus de bayesian_scores(...), contenant au moins:
# ["numero", "proba_bayesienne"]

def weighted_sample_without_replacement(nums, weights, k, rng):
    """
    Tirage sans remise, pondéré.
    Méthode: on transforme les poids en probabilités et on itère.
    (Plus simple/robuste que des hacks, et largement suffisant ici.)
    """
    nums = np.array(nums)
    weights = np.array(weights, dtype=float)
    chosen = []

    available = np.ones(len(nums), dtype=bool)

    for _ in range(k):
        w = weights.copy()
        w[~available] = 0.0
        if w.sum() == 0:
            # fallback uniforme sur ce qui reste
            idxs = np.where(available)[0]
            pick = rng.choice(idxs)
        else:
            p = w / w.sum()
            pick = rng.choice(len(nums), p=p)
        chosen.append(nums[pick])
        available[pick] = False

    return chosen

def generate_diversified_grids(
    df_boules,
    df_etoiles,
    n_grids=5,
    pool_boules=20,
    pool_etoiles=8,
    max_common_boules=2,   # diversification: au plus 2 boules en commun avec une grille précédente
    max_common_etoiles=1,  # au plus 1 étoile en commun
    seed=42
):
    rng = np.random.default_rng(seed)

    # Pools "priorisés"
    b_pool = df_boules.head(pool_boules)[["numero", "proba_bayesienne"]].copy()
    e_pool = df_etoiles.head(pool_etoiles)[["numero", "proba_bayesienne"]].copy()

    b_nums, b_w = b_pool["numero"].tolist(), b_pool["proba_bayesienne"].tolist()
    e_nums, e_w = e_pool["numero"].tolist(), e_pool["proba_bayesienne"].tolist()

    grids = []
    tries_limit = 500

    for _ in range(n_grids):
        for _try in range(tries_limit):
            boules = sorted(weighted_sample_without_replacement(b_nums, b_w, 5, rng))
            etoiles = sorted(weighted_sample_without_replacement(e_nums, e_w, 2, rng))

            # Contrôle diversification vs grilles déjà générées
            ok = True
            for g in grids:
                common_b = len(set(boules) & set(g["boules"]))
                common_e = len(set(etoiles) & set(g["etoiles"]))
                if common_b > max_common_boules or common_e > max_common_etoiles:
                    ok = False
                    break

            if ok:
                grids.append({"boules": boules, "etoiles": etoiles})
                break
        else:
            # Si on n'arrive pas à respecter les contraintes, on relâche un peu
            grids.append({"boules": boules, "etoiles": etoiles})

    return grids

# --- Exemple d'utilisation ---
# n_grids = 3 ou 5
grids = generate_diversified_grids(
    df_boules=df_boules,
    df_etoiles=df_etoiles,
    n_grids=5,
    pool_boules=20,
    pool_etoiles=8,
    max_common_boules=2,
    max_common_etoiles=1,
    seed=20260116
)

for i, g in enumerate(grids, 1):
    print(f"Grille {i}: Boules {g['boules']} | Etoiles {g['etoiles']}")

from IPython.display import display

df_grids = pd.DataFrame([
    {
        "Grille": i + 1,
        "Boules": ", ".join(str(int(x)) for x in g["boules"]),
        "Étoiles": ", ".join(str(int(x)) for x in g["etoiles"])
    }
    for i, g in enumerate(grids)
])

display(df_grids)



Grille 1: Boules [np.int64(10), np.int64(19), np.int64(20), np.int64(29), np.int64(44)] | Etoiles [np.int64(5), np.int64(7)]
Grille 2: Boules [np.int64(13), np.int64(19), np.int64(21), np.int64(23), np.int64(29)] | Etoiles [np.int64(2), np.int64(9)]
Grille 3: Boules [np.int64(21), np.int64(23), np.int64(37), np.int64(45), np.int64(50)] | Etoiles [np.int64(6), np.int64(7)]
Grille 4: Boules [np.int64(17), np.int64(20), np.int64(21), np.int64(26), np.int64(42)] | Etoiles [np.int64(1), np.int64(7)]
Grille 5: Boules [np.int64(13), np.int64(17), np.int64(20), np.int64(24), np.int64(35)] | Etoiles [np.int64(1), np.int64(2)]


,Grille,Boules,Étoiles
0,1,"10, 19, 20, 29, 44","5, 7"
1,2,"13, 19, 21, 23, 29","2, 9"
2,3,"21, 23, 37, 45, 50","6, 7"
3,4,"17, 20, 21, 26, 42","1, 7"
4,5,"13, 17, 20, 24, 35","1, 2"


In [27]:
# Vérification que les propositions ne soient jamais sorties
ball_cols = [f"boule_{i}" for i in range(1, 6)]
star_cols = [f"etoile_{i}" for i in range(1, 3)]

def normalize_combo(values):
    """Transforme une liste/array en tuple trié d'int Python (ignore NaN)."""
    vals = [int(x) for x in values if pd.notna(x)]
    return tuple(sorted(vals))

# tuples (b1..b5) triés
df["boules_tuple"] = df[ball_cols].apply(lambda r: normalize_combo(r.values), axis=1)
# tuples (e1..e2) triés
df["etoiles_tuple"] = df[star_cols].apply(lambda r: normalize_combo(r.values), axis=1)

# combo complet (boules, etoiles)
df["combo_tuple"] = list(zip(df["boules_tuple"], df["etoiles_tuple"]))

historique_combos = set(df["combo_tuple"])
historique_boules = set(df["boules_tuple"])
historique_etoiles = set(df["etoiles_tuple"])

# Vérifier et filtrer tes grilles générées
def grid_key(g):
    boules = normalize_combo(g["boules"])
    etoiles = normalize_combo(g["etoiles"])
    return (boules, etoiles)

def has_already_occurred(g):
    return grid_key(g) in historique_combos

# Grilles jamais sorties (combo complet 5+2)
grids_never_seen = [g for g in grids if not has_already_occurred(g)]

print(f"{len(grids_never_seen)}/{len(grids)} grilles n'ont jamais été tirées.")
for i, g in enumerate(grids_never_seen, 1):
    print(f"Grille {i}: Boules {[int(x) for x in sorted(g['boules'])]} | Étoiles {[int(x) for x in sorted(g['etoiles'])]}")

# Vérifier doublons que les boules 
def boules_already_seen(g):
    return normalize_combo(g["boules"]) in historique_boules

def etoiles_already_seen(g):
    return normalize_combo(g["etoiles"]) in historique_etoiles

for i, g in enumerate(grids, 1):
    b_seen = boules_already_seen(g)
    e_seen = etoiles_already_seen(g)
    full_seen = has_already_occurred(g)
    print(f"Grille {i}: boules_deja_vues={b_seen} | etoiles_deja_vues={e_seen} | combo_complet_deja_sorti={full_seen}")

# Bonus : régénérer automatiquement jusqu’à avoir N grilles “jamais sorties”
def generate_until_never_seen(generate_fn, target_n=5, max_iter=5000):
    out = []
    seen_keys = set()

    for _ in range(max_iter):
        g = generate_fn()  # doit renvoyer {"boules":[...], "etoiles":[...]}
        k = grid_key(g)
        if k in seen_keys:
            continue
        if k in historique_combos:
            continue
        out.append(g)
        seen_keys.add(k)
        if len(out) >= target_n:
            break

    return out
    

5/5 grilles n'ont jamais été tirées.
Grille 1: Boules [10, 19, 20, 29, 44] | Étoiles [5, 7]
Grille 2: Boules [13, 19, 21, 23, 29] | Étoiles [2, 9]
Grille 3: Boules [21, 23, 37, 45, 50] | Étoiles [6, 7]
Grille 4: Boules [17, 20, 21, 26, 42] | Étoiles [1, 7]
Grille 5: Boules [13, 17, 20, 24, 35] | Étoiles [1, 2]
Grille 1: boules_deja_vues=False | etoiles_deja_vues=True | combo_complet_deja_sorti=False
Grille 2: boules_deja_vues=False | etoiles_deja_vues=True | combo_complet_deja_sorti=False
Grille 3: boules_deja_vues=False | etoiles_deja_vues=True | combo_complet_deja_sorti=False
Grille 4: boules_deja_vues=False | etoiles_deja_vues=True | combo_complet_deja_sorti=False
Grille 5: boules_deja_vues=False | etoiles_deja_vues=True | combo_complet_deja_sorti=False


## Combinaison déjà sortie (numéros) ?

In [28]:
def normalize_combo_str(s: str) -> str:
    # extrait tous les nombres, trie, puis reconstruit au format "1-2-3-4-5"
    nums = sorted(map(int, re.findall(r"\d+", str(s))))
    return "-".join(map(str, nums))

def combo_deja_sortie(combo: str) -> bool:
    target = normalize_combo_str(combo)
    # normalise la colonne au même format (au cas où il y a des espaces, virgules, etc.)
    col_norm = df["boules_croissant"].astype(str).map(normalize_combo_str)
    return (col_norm == target).any()

# Exemple

# Pour tester, déjà sortie tirage 690 du 2014-02-05, celui détecté en doublon => affiche True 
# combo = "4-30-31-38-42" 

# combo = "6-8-19-30-43"
# combo = "6-8-19-33-40"
combo = "1-13-34-36-47"

if (combo_deja_sortie(combo)): 
    print("La combinaison", combo, " est déjà sortie.")
else:
    print("La combinaison", combo, "n'est jamais sortie.")

La combinaison 1-13-34-36-47 n'est jamais sortie.


## Numéros et étoiles sortis/NON sortis dans les n derniers tirages

In [29]:
numeros_cols = ["boule_1", "boule_2", "boule_3", "boule_4", "boule_5"]
etoiles_cols = ["etoile_1", "etoile_2"]

def construire_grille(valeurs, nb_colonnes):
    """
    Transforme une liste de valeurs en DataFrame sous forme de grille.
    """
    nb_lignes = math.ceil(len(valeurs) / nb_colonnes)
    grille = []

    for i in range(nb_lignes):
        ligne = valeurs[i * nb_colonnes:(i + 1) * nb_colonnes]
        while len(ligne) < nb_colonnes:
            ligne.append("")
        grille.append(ligne)

    return pd.DataFrame(grille)

def styler_grille(grille_df, valeurs_sorties):
    """
    Colore la grille :
    - rouge = valeur sortie dans les n derniers tirages
    - vert = valeur non sortie
    """
    def couleur_case(val):
        if val == "":
            return ""
        try:
            v = int(val)
        except (ValueError, TypeError):
            return ""

        if v in valeurs_sorties:
            return (
                "background-color: #c62828; "
                "color: white; "
                "font-weight: bold; "
                "text-align: center; "
                "border: 1px solid #999;"
            )
        else:
            return (
                "background-color: #2e7d32; "
                "color: white; "
                "font-weight: bold; "
                "text-align: center; "
                "border: 1px solid #999;"
            )

    return (
        grille_df.style
        .map(couleur_case)
        .set_properties(**{
            "width": "48px",
            "height": "48px",
            "text-align": "center",
            "font-size": "14px"
        })
        .hide(axis="index")
        .hide(axis="columns")
    )

def lasts(n: int):
    derniers = df.tail(n).copy()

    # Sécurise les types numériques
    for col in numeros_cols + etoiles_cols:
        derniers[col] = pd.to_numeric(derniers[col], errors="coerce")

    # Sortis
    numeros_out = {
        int(x) for x in derniers[numeros_cols].values.ravel()
        if pd.notna(x)
    }
    etoiles_out = {
        int(x) for x in derniers[etoiles_cols].values.ravel()
        if pd.notna(x)
    }

    print(f"Numéros sortis dans les {n} derniers tirages  : {sorted(numeros_out)}")
    print(f"Étoiles sorties dans les {n} derniers tirages : {sorted(etoiles_out)}")

    # Non sortis
    numeros_all = set(range(1, 51))
    etoiles_all = set(range(1, 13))

    numeros_not = numeros_all - numeros_out
    etoiles_not = etoiles_all - etoiles_out

    print(f"Numéros NON sortis dans les {n} derniers tirages : {sorted(numeros_not)}")
    print(f"Étoiles NON sorties dans les {n} derniers tirages : {sorted(etoiles_not)}")

    # --- Affichage grille numéros ---
    print(f"\nGrille des numéros (1 à 50) sur les {n} derniers tirages :")
    grille_numeros = construire_grille(list(range(1, 51)), 10)
    display(styler_grille(grille_numeros, numeros_out))

    # --- Affichage grille étoiles ---
    print(f"Grille des étoiles (1 à 12) sur les {n} derniers tirages :")
    grille_etoiles = construire_grille(list(range(1, 13)), 6)
    display(styler_grille(grille_etoiles, etoiles_out))

# -- fin fonction ----------------------

lasts(10)
lasts(5)

Numéros sortis dans les 10 derniers tirages  : [1, 2, 3, 4, 8, 9, 10, 11, 13, 14, 16, 17, 19, 20, 22, 23, 25, 26, 28, 29, 30, 31, 34, 36, 37, 38, 40, 41, 42, 44, 45, 46, 47, 49]
Étoiles sorties dans les 10 derniers tirages : [1, 3, 4, 5, 6, 7, 8, 9, 11, 12]
Numéros NON sortis dans les 10 derniers tirages : [5, 6, 7, 12, 15, 18, 21, 24, 27, 32, 33, 35, 39, 43, 48, 50]
Étoiles NON sorties dans les 10 derniers tirages : [2, 10]

Grille des numéros (1 à 50) sur les 10 derniers tirages :


1,2,3,4,5,6,7,8,9,10
11,12,13,14,15,16,17,18,19,20
21,22,23,24,25,26,27,28,29,30
31,32,33,34,35,36,37,38,39,40
41,42,43,44,45,46,47,48,49,50


Grille des étoiles (1 à 12) sur les 10 derniers tirages :


1,2,3,4,5,6
7,8,9,10,11,12


Numéros sortis dans les 5 derniers tirages  : [2, 3, 4, 8, 9, 17, 19, 20, 25, 26, 29, 30, 31, 34, 37, 40, 41, 42, 45, 46, 47]
Étoiles sorties dans les 5 derniers tirages : [1, 5, 6, 8, 9, 11]
Numéros NON sortis dans les 5 derniers tirages : [1, 5, 6, 7, 10, 11, 12, 13, 14, 15, 16, 18, 21, 22, 23, 24, 27, 28, 32, 33, 35, 36, 38, 39, 43, 44, 48, 49, 50]
Étoiles NON sorties dans les 5 derniers tirages : [2, 3, 4, 7, 10, 12]

Grille des numéros (1 à 50) sur les 5 derniers tirages :


1,2,3,4,5,6,7,8,9,10
11,12,13,14,15,16,17,18,19,20
21,22,23,24,25,26,27,28,29,30
31,32,33,34,35,36,37,38,39,40
41,42,43,44,45,46,47,48,49,50


Grille des étoiles (1 à 12) sur les 5 derniers tirages :


1,2,3,4,5,6
7,8,9,10,11,12


## Affichage du dataframe complet

In [30]:
# Pour vérifications => décommenter si besoin
# with pd.option_context("display.max_rows", None, "display.max_columns", None):
#    df["date_tirage"] = df["date_tirage"].dt.strftime("%d/%m/%Y")
#    display(df)

# Analyse structure tirages


In [31]:
import csv
from collections import Counter
from typing import Iterable

CSV_PATH = "csv/global.csv"
BOULE_COLS = ["boule_1", "boule_2", "boule_3", "boule_4", "boule_5"]


def detect_csv_delimiter(path: str, encoding: str = "utf-8-sig") -> str:
    """Détecte le séparateur CSV pour accepter global.csv (,) et les fichiers FDJ (;)."""
    with open(path, newline="", encoding=encoding) as f:
        sample = f.read(4096)

    try:
        return csv.Sniffer().sniff(sample, delimiters=",;").delimiter
    except csv.Error:
        first_line = sample.splitlines()[0] if sample else ""
        return ";" if first_line.count(";") > first_line.count(",") else ","


def load_draws(path: str = CSV_PATH) -> list[list[int]]:
    draws = []
    delimiter = detect_csv_delimiter(path)

    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f, delimiter=delimiter)
        missing = [c for c in BOULE_COLS if c not in (reader.fieldnames or [])]
        if missing:
            raise ValueError(
                f"Colonnes manquantes dans le CSV: {missing}. "
                f"Colonnes détectées: {reader.fieldnames or []}. "
                f"Séparateur détecté: {delimiter!r}."
            )

        for row in reader:
            draws.append([int(row[c]) for c in BOULE_COLS])
    return draws


def consecutive_sequences(numbers: Iterable[int]) -> list[tuple[int, int]]:
    ordered = sorted(numbers)
    if not ordered:
        return []

    sequences = []
    start = ordered[0]
    prev = ordered[0]

    for n in ordered[1:]:
        if n == prev + 1:
            prev = n
            continue
        if prev > start:
            sequences.append((start, prev))
        start = prev = n

    if prev > start:
        sequences.append((start, prev))
    return sequences


def decade_bucket(n: int) -> str:
    low = ((n - 1) // 10) * 10 + 1
    high = min(low + 9, 50)
    return f"{low:02d}-{high:02d}"


def analyze_standard_structure(draws: list[list[int]]) -> dict:
    per_draw_sequences = []
    decade_patterns = []
    all_balls = []

    for numbers in draws:
        all_balls.extend(numbers)

        seqs = consecutive_sequences(numbers)
        per_draw_sequences.append(len(seqs))

        buckets = sorted(decade_bucket(n) for n in numbers)
        decade_patterns.append(tuple(buckets))

    bucket_distribution = Counter(decade_bucket(n) for n in all_balls)

    return {
        "total_tirages": len(draws),
        "freq_nb_suites": Counter(per_draw_sequences),
        "top_patterns_dizaines": Counter(decade_patterns).most_common(10),
        "distribution_dizaines": dict(sorted(bucket_distribution.items())),
    }


def print_report(report: dict) -> None:
    print("=== Structure standard des tirages (boules principales) ===")
    print(f"Nombre total de tirages analysés : {report['total_tirages']}")

    if report["total_tirages"] == 0:
        print("Aucun tirage à analyser.")
        return

    print("\n1) Nombre de suites consécutives par tirage")
    for nb_suites, freq in sorted(report["freq_nb_suites"].items()):
        pct = (freq / report["total_tirages"]) * 100
        print(f"  - {nb_suites} suite(s): {freq} tirages ({pct:.1f}%)")

    print("\n2) Patterns de dizaines les plus fréquents (Top 10)")
    for pattern, freq in report["top_patterns_dizaines"]:
        print(f"  - {pattern}: {freq}")

    print("\n3) Répartition globale par dizaine")
    total_balls = report["total_tirages"] * len(BOULE_COLS)
    for decade, freq in report["distribution_dizaines"].items():
        pct = (freq / total_balls) * 100
        print(f"  - {decade}: {freq} ({pct:.1f}%)")


if __name__ == "__main__":
    draws = load_draws()
    result = analyze_standard_structure(draws)
    print_report(result)

=== Structure standard des tirages (boules principales) ===
Nombre total de tirages analysés : 1944

1) Nombre de suites consécutives par tirage
  - 0 suite(s): 1269 tirages (65.3%)
  - 1 suite(s): 634 tirages (32.6%)
  - 2 suite(s): 41 tirages (2.1%)

2) Patterns de dizaines les plus fréquents (Top 10)
  - ('01-10', '11-20', '21-30', '31-40', '41-50'): 89
  - ('01-10', '11-20', '11-20', '31-40', '41-50'): 52
  - ('01-10', '11-20', '31-40', '41-50', '41-50'): 48
  - ('01-10', '21-30', '21-30', '31-40', '41-50'): 48
  - ('11-20', '21-30', '31-40', '31-40', '41-50'): 47
  - ('01-10', '21-30', '31-40', '31-40', '41-50'): 46
  - ('01-10', '01-10', '11-20', '21-30', '41-50'): 44
  - ('01-10', '01-10', '11-20', '31-40', '41-50'): 42
  - ('01-10', '11-20', '21-30', '21-30', '41-50'): 42
  - ('01-10', '11-20', '21-30', '31-40', '31-40'): 41

3) Répartition globale par dizaine
  - 01-10: 1917 (19.7%)
  - 11-20: 1981 (20.4%)
  - 21-30: 1989 (20.5%)
  - 31-40: 1882 (19.4%)
  - 41-50: 1951 (20.1%)